# Phase 5: Stacking Ensemble Development & Comprehensive Model Evaluation

**Project:** AI-Based Predictive Maintenance System for CNC Machines in Automotive Manufacturing Using Industrial Sensor Data  
**Objective:** Build and empirically evaluate a leakage-free Stacking Ensemble combining Decision Tree, Random Forest, XGBoost, and LightGBM base learners with a Logistic Regression meta-learner using 5-fold Stratified Cross-Validation.

---

## 1. Setup, Environment, and Data Loading

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from config import (
    PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR, METRICS_DIR,
    RANDOM_STATE, TARGET_COL
)
from src.train_models import load_processed_data, evaluate_model, plot_confusion_matrix
from src.ensemble import (
    build_base_estimators,
    build_stacking_ensemble,
    plot_final_model_comparison,
    run_stacking_pipeline
)

# Load processed SMOTE training data and untouched test data
X_train, y_train, X_test, y_test, feature_names = load_processed_data()
print(f"SMOTE Training Set: {X_train.shape[0]:,} observations, {X_train.shape[1]} features")
print(f"Untouched Test Set: {X_test.shape[0]:,} observations, {X_test.shape[1]} features")
print(f"Test Set Class Counts: {y_test.value_counts().to_dict()} ({y_test.sum()} failures)")

## 2. Review of Phase 4 Baseline Model Results

Inspects the existing baseline metrics across Decision Tree, Random Forest, XGBoost, and LightGBM.

In [ ]:
phase4_df = pd.read_csv(PROJECT_ROOT / "outputs" / "metrics" / "model_performance.csv")
display(phase4_df)

## 3. Stacking Ensemble Architecture & Leakage-Free Design

- **Base Estimators:**
  1. `DecisionTreeClassifier` (Interpretable baseline)
  2. `RandomForestClassifier` (Bagging ensemble)
  3. `XGBClassifier` (Gradient boosted trees)
  4. `LGBMClassifier` (Leaf-wise gradient boosted trees)
- **Meta-Learner:** `LogisticRegression`
- **Out-of-Fold Generation:** 5-Fold `StratifiedKFold` cross-validation across training data ($N=15,458$).
- **Test Set Isolation:** Test set ($N=2,000$) is never seen during base model training, out-of-fold cross-validation, or meta-learner fitting.

In [ ]:
base_estimators = build_base_estimators(RANDOM_STATE)
stacking_model = build_stacking_ensemble(base_estimators, n_splits=5, random_state=RANDOM_STATE)
print("Stacking ensemble initialized with 4 base models and Logistic Regression meta-learner.")

## 4. Train and Serialize Stacking Ensemble

In [ ]:
print("Training Stacking Classifier...")
stacking_model.fit(X_train, y_train)
ensemble_path = MODELS_DIR / "stacking_ensemble.joblib"
joblib.dump(stacking_model, ensemble_path)
print(f"Serialized stacking model saved to: {ensemble_path}")

## 5. Evaluate Stacking Ensemble on Untouched Test Set

In [ ]:
eval_res = evaluate_model(stacking_model, X_test, y_test)
print(f"Accuracy:  {eval_res['accuracy']:.4f}")
print(f"Precision: {eval_res['precision']:.4f}")
print(f"Recall:    {eval_res['recall']:.4f}")
print(f"F1-Score:  {eval_res['f1_score']:.4f}")
print(f"ROC-AUC:   {eval_res['roc_auc']:.4f}")
print(f"PR-AUC:    {eval_res['pr_auc']:.4f}")
print(f"Confusion Matrix: TN={eval_res['tn']}, FP={eval_res['fp']}, FN={eval_res['fn']}, TP={eval_res['tp']}")

plot_confusion_matrix(eval_res["confusion_matrix"], "Stacking Ensemble", FIGURES_DIR / "confusion_matrix_stacking_ensemble.png")

## 6. Complete 5-Model Performance Comparison

In [ ]:
summary = run_stacking_pipeline()
comparison_df = pd.read_csv(PROJECT_ROOT / "outputs" / "tables" / "final_model_comparison.csv")
display(comparison_df)

## 7. Head-to-Head Analysis: LightGBM vs. Stacking Ensemble

In [ ]:
lgbm = comparison_df[comparison_df["Model"] == "LightGBM"].iloc[0]
stack = comparison_df[comparison_df["Model"] == "Stacking Ensemble"].iloc[0]

head_to_head = pd.DataFrame([
    {"Metric": "Recall (Positive Class 1)", "LightGBM": lgbm["Recall"], "Stacking Ensemble": stack["Recall"], "Difference (Stack - LGBM)": round(stack["Recall"] - lgbm["Recall"], 4)},
    {"Metric": "False Negatives (Missed Failures)", "LightGBM": lgbm["False Negative"], "Stacking Ensemble": stack["False Negative"], "Difference (Stack - LGBM)": stack["False Negative"] - lgbm["False Negative"]},
    {"Metric": "Precision", "LightGBM": lgbm["Precision"], "Stacking Ensemble": stack["Precision"], "Difference (Stack - LGBM)": round(stack["Precision"] - lgbm["Precision"], 4)},
    {"Metric": "False Positives (False Alarms)", "LightGBM": lgbm["False Positive"], "Stacking Ensemble": stack["False Positive"], "Difference (Stack - LGBM)": stack["False Positive"] - lgbm["False Positive"]},
    {"Metric": "F1-Score", "LightGBM": lgbm["F1-Score"], "Stacking Ensemble": stack["F1-Score"], "Difference (Stack - LGBM)": round(stack["F1-Score"] - lgbm["F1-Score"], 4)},
    {"Metric": "PR-AUC", "LightGBM": lgbm["PR-AUC"], "Stacking Ensemble": stack["PR-AUC"], "Difference (Stack - LGBM)": round(stack["PR-AUC"] - lgbm["PR-AUC"], 4)},
    {"Metric": "ROC-AUC", "LightGBM": lgbm["ROC-AUC"], "Stacking Ensemble": stack["ROC-AUC"], "Difference (Stack - LGBM)": round(stack["ROC-AUC"] - lgbm["ROC-AUC"], 4)},
    {"Metric": "Accuracy", "LightGBM": lgbm["Accuracy"], "Stacking Ensemble": stack["Accuracy"], "Difference (Stack - LGBM)": round(stack["Accuracy"] - lgbm["Accuracy"], 4)}
])
display(head_to_head)

## 8. Meta-Learner Coefficient Weights

Inspects the linear coefficients assigned by the Logistic Regression meta-learner to each base model's probability predictions.

In [ ]:
meta_clf = stacking_model.final_estimator_
coef_df = pd.DataFrame({
    "Base Estimator": [name for name, _ in base_estimators],
    "Meta-Learner Weight (Coef)": meta_clf.coef_[0]
}).sort_values("Meta-Learner Weight (Coef)", ascending=False)
display(coef_df)